<a href="https://colab.research.google.com/github/marcio-antonio/NumericalModeling_wingPlane/blob/main/Project_numericalModeling_wingPlane_Composite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
***********************************************************************
* SAE AERODESIGN - CARBON FIBER COMPOSITE SPAR
* STAGE 5 - 3D HOLLOW CYLINDER COMPOSITE SHELL MODEL (COQ4)
* ORTHOTROPIC MATERIAL & TSAI-HILL FAILURE EVALUATION
***********************************************************************

OPTI DIME 3 ELEM QUA4 MODE TRID ;

* ------------------------------------------------
* 1. GEOMETRY & MESH (3D Hollow Cylinder Surface)
* ------------------------------------------------

* Spar Length and Outer Radius (Diameter D = 0.20 m)
L = 10.0 ;
R = 0.10 ;

* Origin / Center of root circle at X = 0
C0 = 0.0 0.0 0.0 ;

* 4 key points on the root circle profile at X = 0
P_TOP   = 0.0 R       0.0 ;
P_RIGHT = 0.0 0.0     R ;
P_BOT   = 0.0 (0.0-R) 0.0 ;
P_LEFT  = 0.0 0.0     (0.0-R) ;

* Create 4 circular arcs forming a closed root circle (8 elements per arc)
ARC1 = CERC 8 P_TOP   C0 P_RIGHT ;
ARC2 = CERC 8 P_RIGHT C0 P_BOT ;
ARC3 = CERC 8 P_BOT   C0 P_LEFT ;
ARC4 = CERC 8 P_LEFT  C0 P_TOP ;

* Combine arcs into a single root ring boundary
RING_ROOT = ARC1 ET ARC2 ET ARC3 ET ARC4 ;

* Extrude the root ring along the X-axis by length L (50 elements along span)
VEC_EXT = L 0.0 0.0 ;
SPAR_SURF = RING_ROOT TRAN 50 VEC_EXT ;

* Define key points for boundary conditions and loads
PW    = SPAR_SURF POIN 'PROC' (3.0 R 0.0) ;
P_TIP = SPAR_SURF POIN 'PROC' (10.0 R 0.0) ;

* ------------------------------------------------
* 2. CARBON/EPOXY LAMINA MATERIAL PROPERTIES
* ------------------------------------------------

E1    = 135.E9 ;
E2    = 10.E9 ;
NU12  = 0.30 ;
G12   = 5.0E9 ;
G13   = 5.0E9 ;
G23   = 3.8E9 ;
RHO_C = 1550. ;

* Tube Wall Thickness = 4 mm
THICK = 0.004 ;

* Material Strengths for Tsai-Hill Criterion (Pa)
XT = 1500.E6 ;
XC = 1200.E6 ;
YT = 50.E6 ;
YC = 200.E6 ;
S  = 70.E6 ;

* ------------------------------------------------
* 3. MODEL & MATERIAL DEFINITION
* ------------------------------------------------

MOD_COMP = MODE SPAR_SURF 'MECANIQUE' 'ELASTIQUE' 'ORTHOTROPE' 'COQ4' ;

* Fiber direction vector running along the tube length (X-axis)
VX = 1.0 0.0 0.0 ;

* Assign orthotropic material properties
MAT_COMP = MATE MOD_COMP 'YG1'  E1   'YG2'  E2   'NU12' NU12
                         'G12'  G12  'G13'  G13  'G23'  G23
                         'RHO'  RHO_C 'EPAI' THICK
                         'DIRECTION' VX ;

* Stiffness matrix and root clamping boundary condition
RIG = RIGI MOD_COMP MAT_COMP ;
BLOC = BLOQ 'DEPL' 'ROTA' RING_ROOT ;
RIGB = RIG ET BLOC ;

* ------------------------------------------------
* 4. APPLIED LOADS
* ------------------------------------------------

* Surface force density based on projected frontal width (Diameter = 2*R)
PRESS_Y = 6000. / (2.0 * R) ;
VY_FORCE = 0.0 PRESS_Y 0.0 ;

* Apply distributed surface force density
LOAD_Q = FSUR 'COQU' MOD_COMP VY_FORCE ;

* Concentrated Weight Load at X = 3 m
MASS = 250. ;
G = 9.81 ;
W = MASS * G ;
LOAD_W = FORC 'FY' (0.0 - W) PW ;

* Combined Total Load Vector
LOAD = LOAD_Q ET LOAD_W ;

* ------------------------------------------------
* 5. SOLVE
* ------------------------------------------------

U_COMP = RESO RIGB LOAD ;

* ------------------------------------------------
* 6. POST-PROCESSING & STRESS CALCULATIONS
* ------------------------------------------------

* Extract Tip Vertical Deflection
UY_TIP_COMP = EXTR U_COMP 'UY' P_TIP ;

* Compute internal shell efforts (N and M components)
SIG_COMP = SIGM MOD_COMP MAT_COMP U_COMP ;
CHPO_SIG = CHAN 'CHPO' MOD_COMP SIG_COMP ;

* Extract Membrane Forces (N/m) and Bending Moments (N.m/m)
N_11 = EXCO 'N11' CHPO_SIG ;
M_11 = EXCO 'M11' CHPO_SIG ;

N_22 = EXCO 'N22' CHPO_SIG ;
M_22 = EXCO 'M22' CHPO_SIG ;

N_12 = EXCO 'N12' CHPO_SIG ;
M_12 = EXCO 'M12' CHPO_SIG ;

* Geometric conversion factors for shell outer fibers
INV_T  = 1.0 / THICK ;
INV_T2 = 6.0 / (THICK * THICK) ;

* True Total Outer Fiber Stresses: Sigma = (N / t) + (6 * M / t^2)
SIG_11 = (N_11 * INV_T) + (M_11 * INV_T2) ;
SIG_22 = (N_22 * INV_T) + (M_22 * INV_T2) ;
TAU_12 = (N_12 * INV_T) + (M_12 * INV_T2) ;

* Extract Peak Stresses across mesh
SIG11_MAX = MAXI SIG_11 ;
SIG22_MAX = MAXI SIG_22 ;
TAU12_MAX = MAXI TAU_12 ;

* Tsai-Hill Failure Index calculation at maximum stress point
TH_TERM1 = (SIG11_MAX / XT) * (SIG11_MAX / XT) ;
TH_TERM2 = (SIG22_MAX / YT) * (SIG22_MAX / YT) ;
TH_TERM3 = (TAU12_MAX / S)  * (TAU12_MAX / S)  ;
TSAI_HILL_MAX = TH_TERM1 + TH_TERM2 + TH_TERM3 ;

MESS ' ';
MESS '================================================';
MESS '3D HOLLOW CYLINDER COMPOSITE SPAR RESULTS';
MESS '================================================';
MESS 'MAX TIP DEFLECTION Uy (m)       = ' UY_TIP_COMP ;
MESS 'MAX FIBER STRESS Sig11 (Pa)    = ' SIG11_MAX ;
MESS 'MAX TRANSVERSE STRESS Sig22 (Pa) = ' SIG22_MAX ;
MESS 'MAX IN-PLANE SHEAR Tau12 (Pa)  = ' TAU12_MAX ;
MESS '------------------------------------------------';
MESS 'ESTIMATED TSAI-HILL INDEX       = ' TSAI_HILL_MAX ;
MESS '================================================';

* ------------------------------------------------
* 7. GRAPHICAL VISUALIZATION (HORIZONTAL SIDE VIEW)
* ------------------------------------------------

* Place camera far on Z-axis centered at X = 5.0, Y = 0.0
* This projects the 3D cylinder as a horizontal beam in the XY plane
EYE = 5.0 0.0 100.0 ;
OPTI 'OEIL' EYE ;

* 1. Deformed shape plot (Horizontal View)
DEF0 = DEFO SPAR_SURF U_COMP 0. ;
DEF1 = DEFO SPAR_SURF U_COMP 1. 'ROUG' ;
TRAC (DEF0 ET DEF1) 'TITR' '1. 3D Cylinder - Deformed vs Initial (Side View)' ;

* 2. Fiber Stress Contour Field (Horizontal View)
TRAC SIG_11 SPAR_SURF 'TITR' '2. Fiber Stress Field Sig11 [Pa]' ;

* 3. Transverse Matrix Stress Field (Horizontal View)
TRAC SIG_22 SPAR_SURF 'TITR' '3. Transverse Stress Field Sig22 [Pa]' ;

* 4. In-Plane Shear Stress Field (Horizontal View)
TRAC TAU_12 SPAR_SURF 'TITR' '4. In-Plane Shear Stress Field Tau12 [Pa]' ;

* ============================================================
* OVALIZATION CROSS-SECTION VIEW (ON-SCREEN VIEW)
* ============================================================
* Set camera looking down the X axis
OPTI 'OEIL' (10.0 0.0 0.0) ;

* Switch driver to interactive screen window
OPTI 'TRAC' 'OPEN' ;

* 1. Define 3 reference points lying on the X = 0 plane
P_ORIG = 0. 0. 0. ;
P_YAX  = 0. 1. 0. ;
P_ZAX  = 0. 0. 1. ;

* 2. Select nodes near the plane
nodes_root = SPAR_SURF POIN 'PLAN' P_ORIG P_YAX P_ZAX 0.01 ;

* 3. Extract element ring
root_section = SPAR_SURF ELEM 'APPU' 'LARG' nodes_root ;

* 4. Compute deformed mesh (20x amplification)
DEF_ROOT_0 = DEFO root_section U_COMP 0. ;
DEF_ROOT_1 = DEFO root_section U_COMP 20. 'ROUG' ;

* 5. Display on screen
TRAC (DEF_ROOT_0 ET DEF_ROOT_1) 'TITR' 'Cross-Section Ovalization at Root (20x Ampli)' ;

* PAUSE halts execution so the window stays open on screen!
* Press Enter in the terminal to close the window and finish.
PAUSE ;

FIN ;


FIN ;